# 04 — Audio

How well the system answers questions about recordings: Whisper, segment packing, retrieval and answering, on SLUE-SQA-5 (Shon et al., 2023), real people reading Wikipedia articles aloud, with answers timed in the audio.

The same questions also go through the pipeline with the benchmark's reference transcript in place of Whisper's, so the difference between the two is what speech recognition errors cost.

**Read the support labels first.** SLUE-SQA-5 paired each question with a clip that contains the answer *string*. Reading all 300 pairs (`data/eval/spoken_qa_support_labels.json`) found that only 66 clips answer their question and 28 imply the answer; the other 206 are coincidences. The numbers that mean something are on the `answered_by_clip` rows.

**Running it.** Every section below is the evaluation's own code. With `RUN = False`
(the default) nothing is recomputed: the results saved in `data/eval/` are loaded
and shown. Set `RUN = True` in the first code cell to measure again, which
overwrites those files. The first run downloads the test split (5.7 GB) to the Hugging Face cache; Whisper base transcribes the 2.9 hours of audio in about 9 minutes on the CPU.

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath("") if os.path.basename(os.path.abspath("")) == "notebooks"
                else os.path.join(os.path.abspath(""), "notebooks"))
from eval_common import repo_root  # noqa: E402

REPO_ROOT = repo_root()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

# The application's settings (backend/service.py). They are read when the
# pipeline modules are imported, so they are set before anything else.
os.environ.setdefault("SA_EMBEDDER", "bge-small")

# False: show the results saved in data/eval. True: run the evaluation again
# and overwrite them (the run time is given at the top of the notebook).
RUN = False

In [2]:
import json

import pandas as pd

EVAL_DIR = REPO_ROOT / "data" / "eval"


def saved(name):
    """A results file from data/eval."""
    return json.loads((EVAL_DIR / name).read_text(encoding="utf-8"))

## SLUE-SQA-5

Runs the audio path on SLUE-SQA-5 (Shon et al., 2023), the spoken-document
half of the evaluation that the PDF benchmarks cannot cover.

#### Why this benchmark
Every other evaluation in this project reads text. The audio path adds a stage
none of them exercise — Whisper — and its errors reach retrieval and the
answer before anything else sees them. SLUE-SQA-5 is built for exactly that:

```text
  documents   real people reading Spoken Wikipedia articles (about 400
              speakers), cut into roughly 40-second clips, with a reference
              transcript and the time of every word
  questions   from SQuAD, Natural Questions, TriviaQA, WebQuestions and
              CuratedTREC; the answer is a span, marked in the audio to the
              hundredth of a second
```

A short span means the answers are scored with SQuAD's exact match and token
F1, the metric family QASPER already uses here; a timed span means retrieval
can be scored by whether a retrieved chunk covers the moment the answer is
spoken, which is Recall@k for a recording.

#### How the recordings are built
A 40-second clip is shorter than one chunk, so indexing clips one by one would
never exercise _pack_audio's segment packing or pause breaks. Clips cut from
the same article are therefore joined, in article order and with a second of
silence between them, into one recording per article, and the recordings of
every chosen article go into one index — the way a student's subject holds
several lectures. A question has to find the right moment in the right
recording among all of them.

Each recording goes through loader.load_audio_segments, which is what an
upload goes through: Whisper and its own segmentation, then
preprocessor.preprocess and _pack_audio, the embedder and the hybrid
retriever. The application uses Whisper "base"; --whisper small|medium|turbo
runs the same path with a larger model, to measure what recognition quality
buys against transcription time.

#### Two runs of the same pipeline
```text
  whisper     the recordings transcribed by Whisper, as in the application
  reference   the benchmark's reference transcript, cut into segments at
              sentence ends with the benchmark's word times, and put through
              the same packing, embedding, retrieval and answering
```

The pipeline after transcription is identical, so the gap between the two is
what speech recognition errors alone cost.

#### Scoring
```text
  Recall@k    a retrieved chunk is from the right recording and its time
              range overlaps the answer's span (TOLERANCE seconds of slack
              for Whisper's segment boundaries)
  MRR@5       reciprocal rank of the first such chunk
  in context  the gold answer's words appear in the chunks given to the model
  EM / F1     SQuAD's exact match and token F1 (the vendored QASPER
              evaluator's copy), best over the answer's two written forms:
              the benchmark's spoken form ("two thousand five") and the words
              actually printed in the article at that moment ("2005") —
              otherwise every numeric answer scores zero for being written in
              digits
```

In [3]:
# The script's command-line options, as it would have read them.
QUESTIONS = 300          # the first 150 are the original 150-question sample
WHISPER = "base"         # the application's model; notebook 08 compares small and turbo
sys.argv = ["notebook", "--questions", str(QUESTIONS), "--whisper", WHISPER]

In [4]:
import json
import os
import random
import re
import statistics
import subprocess
import sys
import time
import wave
from collections import defaultdict
from pathlib import Path

ROOT = REPO_ROOT
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import faiss  # noqa: E402
import numpy as np  # noqa: E402
import pyarrow.parquet as pq  # noqa: E402
from huggingface_hub import hf_hub_download  # noqa: E402

from backend.pipeline import generator, sparse as sparse_module  # noqa: E402
from backend.pipeline.embedder import embed, model_key  # noqa: E402
from backend.pipeline.loader import load_audio_segments  # noqa: E402
from backend.pipeline.preprocessor import preprocess  # noqa: E402
from backend.pipeline.retriever import DENSE_WEIGHT, retrieve  # noqa: E402
from backend.scripts.vendor.qasper_evaluator import (  # noqa: E402
    normalize_answer, token_f1_score)

EVAL_DIR = ROOT / "data" / "eval"
WORK_DIR = Path(os.environ.get(
    "SA_SLUE_DIR", Path.home() / ".cache" / "student-assistant" / "slue_sqa5"))

REPO = "asapp/slue-phase-2"
SHARDS = [f"sqa5/test-{i:05d}-of-00014.parquet" for i in range(14)]
META_COLUMNS = ["question_id", "raw_question_text", "document_id",
                "word2time", "answer_spans", "normalized_document_text"]

SEED = 13
CHUNKING = "sentence"
SAMPLE_RATE = 16000
GAP_SECONDS = 1.0            # silence between clips joined into a recording
TOLERANCE = 0.5              # slack on the answer span when matching chunks
RECALL_AT = (1, 3, 5)
REFERENCE_SEGMENT_WORDS = 30  # a Whisper segment is rarely longer
MAX_PER_ARTICLE = 12         # one long article must not be the whole sample
SUPPORT_LABELS = EVAL_DIR / "spoken_qa_support_labels.json"

In [5]:
def arg(flag, default):
    return type(default)(sys.argv[sys.argv.index(flag) + 1]) if flag in sys.argv else default

In [6]:
QUESTIONS = arg("--questions", 150)
TOP_K = arg("--k", 3)
WHISPER = arg("--whisper", "base")    # the application's model
MODEL_TAG = ("" if generator.MODEL_NAME == "Qwen/Qwen2.5-1.5B-Instruct"
             else "_" + generator.MODEL_NAME.split("/")[-1].lower()
             .replace("-instruct", "").replace(":", "-"))
OUT_PATH = EVAL_DIR / (f"spoken_qa_{QUESTIONS}q" + MODEL_TAG
                       + ("" if WHISPER == "base" else f"_whisper-{WHISPER}")
                       + ("" if TOP_K == 3 else f"_k{TOP_K}") + ".json")

In [7]:
# ---------------------------------------------------------------- the sample

def shard_paths():
    return [hf_hub_download(REPO, name, repo_type="dataset") for name in SHARDS]

In [8]:
def article_and_position(document_id):
    """'Martin_Luther_29' -> ('Martin_Luther', 29)."""
    article, _, position = document_id.rpartition("_")
    return article, int(position)

In [9]:
def choose(paths):
    """
    Whole articles, in a seeded random order, until QUESTIONS questions are
    in; at most MAX_PER_ARTICLE from any one article. Questions whose clip has
    no timed answer span are skipped — they cannot be scored for retrieval.
    """
    by_article = defaultdict(list)
    for path in paths:
        for row in pq.read_table(path, columns=META_COLUMNS).to_pylist():
            if not row["answer_spans"]["answer"]:
                continue
            by_article[article_and_position(row["document_id"])[0]].append(row)

    rng = random.Random(SEED)
    articles = sorted(by_article)
    rng.shuffle(articles)
    chosen = []
    for article in articles:
        rows = sorted(by_article[article], key=lambda r: r["question_id"])
        rng.shuffle(rows)
        chosen.extend(rows[:MAX_PER_ARTICLE])
        if len(chosen) >= QUESTIONS:
            break
    total = sum(len(v) for v in by_article.values())
    return chosen[:QUESTIONS], total, len(by_article)

In [10]:
def clip_audio(paths, document_ids):
    """The WAV bytes of every clip wanted, read one shard at a time."""
    wanted, found = set(document_ids), {}
    for path in paths:
        table = pq.read_table(path, columns=["document_id", "document_audio"])
        for doc_id, audio in zip(table.column("document_id").to_pylist(),
                                 table.column("document_audio").to_pylist()):
            if doc_id in wanted and doc_id not in found:
                found[doc_id] = audio["bytes"]
        if len(found) == len(wanted):
            break
    return found

In [11]:
def read_wav(raw):
    """
    A clip as 16 kHz mono 16-bit PCM. The benchmark stores 32-bit float WAV,
    which the standard wave module cannot read, so ffmpeg (already required
    by Whisper) converts it.
    """
    result = subprocess.run(
        ["ffmpeg", "-loglevel", "error", "-i", "pipe:0", "-f", "s16le",
         "-ac", "1", "-ar", str(SAMPLE_RATE), "pipe:1"],
        input=raw, capture_output=True, check=True)
    return result.stdout

In [12]:
def build_recordings(rows, paths):
    """
    One WAV per article: its clips in article order with GAP_SECONDS of
    silence between them. Returns {article: {"path", "clips": {doc_id:
    offset_seconds}}} and caches the files, so a re-run skips this.
    """
    clips_of = defaultdict(set)
    for row in rows:
        article, position = article_and_position(row["document_id"])
        clips_of[article].add((position, row["document_id"]))

    out_dir = WORK_DIR / "recordings"
    out_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = out_dir / "manifest.json"
    manifest = (json.loads(manifest_path.read_text(encoding="utf-8"))
                if manifest_path.exists() else {})

    todo = [a for a in clips_of
            if a not in manifest
            or set(manifest[a]["clips"]) != {d for _, d in clips_of[a]}
            or not Path(manifest[a]["path"]).exists()]
    audio = clip_audio(paths, [d for a in todo for _, d in clips_of[a]]) if todo else {}

    for article in todo:
        pcm_parts, offsets, cursor = [], {}, 0.0
        for _, doc_id in sorted(clips_of[article]):
            frames = read_wav(audio[doc_id])
            if pcm_parts:
                pcm_parts.append(b"\x00" * int(GAP_SECONDS * SAMPLE_RATE) * 2)
                cursor += GAP_SECONDS
            offsets[doc_id] = round(cursor, 3)
            pcm_parts.append(frames)
            cursor += len(frames) / (SAMPLE_RATE * 2)
        path = out_dir / (re.sub(r"[^A-Za-z0-9_-]", "_", article) + ".wav")
        with wave.open(str(path), "wb") as w:
            w.setnchannels(1)
            w.setsampwidth(2)
            w.setframerate(SAMPLE_RATE)
            w.writeframes(b"".join(pcm_parts))
        manifest[article] = {"path": str(path), "clips": offsets,
                             "seconds": round(cursor, 1)}
    manifest_path.write_text(json.dumps(manifest, indent=1), encoding="utf-8")
    return {a: manifest[a] for a in clips_of}

In [13]:
# ------------------------------------------------------------ the two inputs

def whisper_segments(article, recording):
    """
    The recording through the upload path's transcriber at WHISPER size,
    cached per article and model with the time it took, so a re-run keeps
    the timing without transcribing again. The application uses "base";
    other sizes are loaded through the same load_audio_segments.
    """
    cache = (WORK_DIR / "transcripts" / WHISPER
             / (Path(recording["path"]).stem + ".json"))
    if cache.exists():
        cached = json.loads(cache.read_text(encoding="utf-8"))
        if cached.get("clips") == recording["clips"]:
            return cached["segments"], cached.get("seconds", 0.0)
    started = time.time()
    segments = load_audio_segments(recording["path"], WHISPER)
    seconds = round(time.time() - started, 1)
    cache.parent.mkdir(parents=True, exist_ok=True)
    cache.write_text(json.dumps({"clips": recording["clips"], "seconds": seconds,
                                 "segments": segments}), encoding="utf-8")
    return segments, seconds

In [14]:
def _join_words(words):
    text = " ".join(words)
    text = re.sub(r" ([.,;:!?%)\]])", r"\1", text)
    return re.sub(r"([(\[$]) ", r"\1", text)

In [15]:
def reference_segments(article, recording, word2time):
    """
    The benchmark's reference transcript in Whisper's shape: segments cut at
    sentence ends (or every REFERENCE_SEGMENT_WORDS words), timed with the
    benchmark's word times shifted to the clip's place in the recording.
    """
    source = Path(recording["path"]).name
    segments = []
    for doc_id, offset in sorted(recording["clips"].items(), key=lambda kv: kv[1]):
        timing = word2time[doc_id]
        words, start, end = [], None, None
        for word, s, e in zip(timing["word"], timing["start_second"],
                              timing["end_second"]):
            words.append(word)
            # Punctuation is untimed (-1 in the benchmark); only spoken
            # words move the segment's start and end.
            if s >= 0:
                start = s if start is None else start
                end = e
            ends = word in (".", "!", "?") or len(words) >= REFERENCE_SEGMENT_WORDS
            if ends and start is not None:
                segments.append({"source_file": source, "kind": "audio",
                                 "text": _join_words(words),
                                 "start": offset + start, "end": offset + end})
                words, start = [], None
        if words and start is not None:
            segments.append({"source_file": source, "kind": "audio",
                             "text": _join_words(words),
                             "start": offset + start, "end": offset + end})
    return segments

In [16]:
# ----------------------------------------------------------------- scoring

def written_answers(row):
    """
    Both forms of the gold answer: the benchmark's spoken form, and the
    article's own words under the answer span ("2005", "San Diego").
    """
    forms = set()
    timing = row["word2time"]
    for answer, s, e in zip(row["answer_spans"]["answer"],
                            row["answer_spans"]["start_second"],
                            row["answer_spans"]["end_second"]):
        forms.add(answer)
        printed = [w for w, ws, we in zip(timing["word"], timing["start_second"],
                                          timing["end_second"])
                   if ws >= s - 0.01 and we <= e + 0.01]
        if printed:
            forms.add(_join_words(printed))
    return sorted(f for f in forms if normalize_answer(f))

In [17]:
def gold_spans(row, recording):
    offset = recording["clips"][row["document_id"]]
    return [(offset + s, offset + e)
            for s, e in zip(row["answer_spans"]["start_second"],
                            row["answer_spans"]["end_second"])]

In [18]:
def covers(chunk, source, spans):
    if getattr(chunk, "source_file", None) != source or chunk.start is None:
        return False
    return any(chunk.start - TOLERANCE <= e and chunk.end + TOLERANCE >= s
               for s, e in spans)

In [19]:
def contains(text, answers):
    haystack = f" {normalize_answer(text)} "
    return any(f" {normalize_answer(a)} " in haystack for a in answers)

In [20]:
def run(label, segments_of, rows, recordings):
    """Index every recording's segments as one subject, ask every question."""
    chunks, per_recording = [], {}
    for article, recording in recordings.items():
        file_chunks = preprocess(segments_of[article],
                                 source_file=Path(recording["path"]).name,
                                 chunking=CHUNKING)
        per_recording[article] = len(file_chunks)
        chunks.extend(file_chunks)
    vectors = embed(chunks)
    index = faiss.IndexFlatL2(vectors.shape[1])
    index.add(np.ascontiguousarray(vectors, dtype=np.float32))
    keywords = sparse_module.build_index(chunks)
    transcript_of = {a: " ".join(s["text"] for s in segments_of[a])
                     for a in recordings}

    depth = max(max(RECALL_AT), TOP_K)
    records = []
    for n, row in enumerate(rows, start=1):
        article = article_and_position(row["document_id"])[0]
        recording = recordings[article]
        source = Path(recording["path"]).name
        spans = gold_spans(row, recording)
        answers = written_answers(row)

        got = retrieve(row["raw_question_text"], index, chunks, k=depth,
                       sparse=keywords, dense_weight=DENSE_WEIGHT)
        first = next((rank for rank, c in enumerate(got, start=1)
                      if covers(c, source, spans)), None)
        context = got[:TOP_K]
        predicted = generator.answer_short(row["raw_question_text"],
                                           [str(c) for c in context])
        records.append({
            "question_id": row["question_id"],
            "question": row["raw_question_text"],
            "gold": answers,
            "predicted": predicted,
            "first_hit_rank": first,
            "in_transcript": contains(transcript_of[article], answers),
            "in_context": contains(" ".join(str(c) for c in context), answers),
            "em": max(float(normalize_answer(predicted) == normalize_answer(a))
                      for a in answers),
            "f1": max(token_f1_score(predicted, a) for a in answers),
        })
        if n % 25 == 0:
            print(f"    {label}: {n}/{len(rows)}")

    summary = {"chunks": len(chunks),
               "chunks_per_recording": statistics.median(per_recording.values()),
               **score(records)}
    return summary, records

In [21]:
def score(records):
    """Retrieval and answer metrics over a list of per-question records."""
    def share(key):
        return statistics.mean(float(r[key]) for r in records)

    return {
        "n": len(records),
        **{f"recall@{k}": statistics.mean(
            float(r["first_hit_rank"] is not None and r["first_hit_rank"] <= k)
            for r in records) for k in RECALL_AT},
        "mrr@5": statistics.mean(
            1 / r["first_hit_rank"] if r["first_hit_rank"] else 0.0
            for r in records),
        "answer_in_transcript": share("in_transcript"),
        "answer_in_context": share("in_context"),
        "exact_match": share("em"),
        "f1": share("f1"),
    }

In [22]:
def by_support(records):
    """
    The same metrics split by whether the question's clip actually answers it
    (SUPPORT_LABELS, read by hand). SLUE-SQA-5 paired each question with a clip
    that contains the answer STRING; in this sample most such pairs are
    coincidences, where no retriever could be expected to find the clip, so
    the headline numbers are only meaningful on the supported subset.
    """
    if not SUPPORT_LABELS.exists():
        return None
    labels = json.loads(SUPPORT_LABELS.read_text(encoding="utf-8"))["labels"]
    groups = {"answered_by_clip": ("yes",),
              "answered_or_implied": ("yes", "partial"),
              "coincidental": ("no",)}
    out = {}
    for name, accepted in groups.items():
        chosen = [r for r in records
                  if labels.get(r["question_id"], {}).get("label") in accepted]
        if chosen:
            out[name] = score(chosen)
    return out

In [23]:
def print_table(results):
    print(f"\n  {'':<22}{'whisper':>10}{'reference':>11}")
    for key in results["whisper"]:
        if key == "by_support":
            continue
        w, r = results["whisper"][key], results["reference"][key]
        fmt = "{:>10.3f}{:>11.3f}" if isinstance(w, float) else "{:>10}{:>11}"
        print(f"  {key:<22}" + fmt.format(w, r))
    for group, w in (results["whisper"].get("by_support") or {}).items():
        r = results["reference"]["by_support"][group]
        print(f"\n  {group} (n={w['n']})")
        for key in ("recall@1", "recall@3", "recall@5", "mrr@5",
                    "answer_in_context", "exact_match", "f1"):
            print(f"    {key:<20}{w[key]:>10.3f}{r[key]:>11.3f}")

In [24]:
def rescore():
    """Recompute the support split on a saved run without re-running it."""
    saved = json.loads(OUT_PATH.read_text(encoding="utf-8"))
    for label, records in saved["records"].items():
        saved["results"][label].setdefault("n", len(records))
        saved["results"][label]["by_support"] = by_support(records)
    OUT_PATH.write_text(json.dumps(saved, indent=2, ensure_ascii=False) + "\n",
                        encoding="utf-8")
    print_table(saved["results"])
    print(f"\n  -> {OUT_PATH.relative_to(ROOT)}")

In [25]:
# -------------------------------------------------------------- spot check

def spotcheck(rows, how_many=30):
    """
    Print question/passage pairs to read. SLUE-SQA-5 pairs each question with
    a clip that contains its answer string, which is not the same as a clip
    that answers it; this is the sample for judging how often that differs.
    """
    rng = random.Random(SEED + 1)
    for row in rng.sample(rows, min(how_many, len(rows))):
        text = row["normalized_document_text"]
        answer = row["answer_spans"]["answer"][0]
        at = text.find(answer)
        window = text[max(0, at - 250): at + len(answer) + 250] if at >= 0 else text[:500]
        print(f"\n[{row['question_id']}] {row['raw_question_text']}\n"
              f"  answer: {answer}   clip: {row['document_id']}\n  ...{window}...")

In [26]:
def main():
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass

    if "--rescore" in sys.argv:
        rescore()
        return

    paths = shard_paths()
    rows, total, articles = choose(paths)
    if "--spotcheck" in sys.argv:
        spotcheck(rows)
        return

    recordings = build_recordings(rows, paths)
    hours = sum(r["seconds"] for r in recordings.values()) / 3600
    print(f"SLUE-SQA-5 test: {len(rows)} of {total} questions, "
          f"{len(recordings)} of {articles} articles, "
          f"{sum(len(r['clips']) for r in recordings.values())} clips, "
          f"{hours:.2f} h of audio")
    print(f"whisper {WHISPER}   model {generator.MODEL_NAME}   "
          f"embedder {model_key()}   k={TOP_K}   "
          f"abstain={generator.ABSTAIN}\n")

    whisper, transcribe_seconds = {}, 0.0
    for n, (article, recording) in enumerate(recordings.items(), start=1):
        whisper[article], seconds = whisper_segments(article, recording)
        transcribe_seconds += seconds
        print(f"  [{n}/{len(recordings)}] {article[:40]:<40} "
              f"{recording['seconds'] / 60:5.1f} min  "
              f"{len(whisper[article]):>4} segments  ({transcribe_seconds:.0f}s)")

    word2time = {r["document_id"]: r["word2time"] for r in rows}
    reference = {a: reference_segments(a, rec, word2time)
                 for a, rec in recordings.items()}

    results, all_records = {}, {}
    for label, segments_of in (("whisper", whisper), ("reference", reference)):
        summary, records = run(label, segments_of, rows, recordings)
        results[label], all_records[label] = summary, records

    for label in results:
        results[label]["by_support"] = by_support(all_records[label])
    print_table(results)

    OUT_PATH.write_text(json.dumps({
        "note": "The audio path on SLUE-SQA-5 (Shon et al., 2023), test split. "
                "Clips of one article are joined into one recording; every "
                "recording is indexed together as one subject. 'whisper' is "
                "the application's path (Whisper, 'base' in the app -> "
                "_pack_audio); 'reference' puts the benchmark's reference "
                "transcript through the same packing, retrieval and answering, "
                "so the gap is what recognition errors cost. Recall@k counts a "
                "chunk from the right recording whose time range overlaps the "
                "answer span. EM/F1 are SQuAD's, best over the spoken and the "
                "printed form of the answer.",
        "questions": len(rows),
        "articles": len(recordings),
        "clips": sum(len(r["clips"]) for r in recordings.values()),
        "audio_hours": round(hours, 2),
        "whisper_model": WHISPER,
        "transcribe_seconds": round(transcribe_seconds, 1),
        "realtime_factor": round(hours * 3600 / transcribe_seconds, 1)
                           if transcribe_seconds else None,
        "model": generator.MODEL_NAME,
        "backend": generator.BACKEND,
        "embedder": model_key(),
        "chunking": CHUNKING,
        "k": TOP_K,
        "abstain": generator.ABSTAIN,
        "tolerance_seconds": TOLERANCE,
        "seed": SEED,
        "results": results,
        "records": all_records,
    }, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"\n  -> {OUT_PATH.relative_to(ROOT)}")

Transcripts are cached per Whisper model in `~/.cache/student-assistant/slue_sqa5`, so a re-run with a model already transcribed takes about 5 minutes.

In [27]:
if RUN:
    main()
else:
    print('RUN is False: showing the saved results below.')

RUN is False: showing the saved results below.


### Results

In [28]:
sq = saved("spoken_qa_300q.json")
print(f"{sq['questions']} questions, {sq['articles']} recordings, {sq['audio_hours']} h, "
      f"Whisper {sq['whisper_model']}, embedder {sq['embedder']}, k={sq['k']}")
metrics = ["recall@1", "recall@3", "recall@5", "mrr@5", "answer_in_context",
           "exact_match", "f1"]
rows = []
for group in ["all"] + list(sq["results"]["whisper"]["by_support"]):
    for transcript in ("whisper", "reference"):
        r = (sq["results"][transcript] if group == "all"
             else sq["results"][transcript]["by_support"][group])
        rows.append({"questions": group, "transcript": transcript, "n": r["n"],
                     **{m: round(r[m], 3) for m in metrics}})
pd.DataFrame(rows).set_index(["questions", "transcript"])

300 questions, 90 recordings, 2.92 h, Whisper base, embedder bge-small, k=3


n  recall@1  recall@3  recall@5  mrr@5  \
questions           transcript                                             
all                 whisper     300     0.530     0.833     0.917  0.683   
                    reference   300     0.530     0.830     0.927  0.679   
answered_by_clip    whisper      66     0.758     0.924     0.924  0.836   
                    reference    66     0.773     0.955     0.985  0.855   
answered_or_implied whisper      94     0.713     0.915     0.936  0.814   
                    reference    94     0.734     0.947     0.968  0.826   
coincidental        whisper     206     0.447     0.796     0.908  0.623   
                    reference   206     0.437     0.777     0.908  0.612   

                                answer_in_context  exact_match     f1  
questions           transcript                                         
all                 whisper                 0.753        0.207  0.283  
                    reference               0.860        0.273  0.331  
answered_by_clip    whisper                 0.697        0.303  0.376  
                    reference               0.955        0.455  0.521  
answered_or_implied whisper                 0.734        0.287  0.373  
                    reference               0.947        0.447  0.513  
coincidental        whisper                 0.762        0.170  0.243  
                    reference               0.820        0.194  0.248

### Where Whisper loses the answer

Clean questions where the reference transcript's answer was right and Whisper's was not.

In [29]:
labels = saved("spoken_qa_support_labels.json")["labels"]
ref = {r["question_id"]: r for r in sq["records"]["reference"]}
pd.DataFrame([{"question": r["question"], "gold": r["gold"][-1],
               "with Whisper": r["predicted"], "with reference": ref[r["question_id"]]["predicted"],
               "answer in Whisper transcript": r["in_transcript"]}
              for r in sq["records"]["whisper"]
              if labels[r["question_id"]]["label"] == "yes"
              and ref[r["question_id"]]["f1"] > r["f1"]])

,question,gold,with Whisper,with reference,answer in Whisper transcript
0,To what level would the polynomial time hierar...,second,"close parenthesis, close parenthesis",second level,True
1,What is the name for a problem that meets Ladn...,np-intermediate,NP intermediate,NP-intermediate,False
2,What eponymous variation of arithmetic present...,presburger arithmetic,MP-complete,Presburger arithmetic,False
3,What is an example of a problem that rests wit...,boolean satisfiability problem,Boolean-Satisfiability,graph isomorphism problem,False
4,Who plotted the relationships between levels o...,kuznets,Simeon Abramovich-Kuznetz,Kuznets,False
5,What is a a developing economy's level of ineq...,kuznets curve,Kuznetz curve,Kuznets curve,False
6,The Song of Simeon canticle is also known by w...,nunc dimittis,Nukh Demetis,Nunc dimittis,False
7,who was assassinated during a visit to sarajev...,archduke franz ferdinand of austria,Gavrilo Princip,Franz Ferdinand,True
8,What day did the Apollo 11 crew return to Earth?,july 24,24,July 24,False
9,"Who wounded Achilles in the heel, leading to h...",paris,"February 18,1546",Paris,True
